# Cap 06 — Memoria y Checkpointing

Damos memoria persistente al grafo usando checkpointers:
- `MemorySaver`: checkpointer in-memory
- `thread_id`: identificador de sesión de conversación
- `get_state_history`: acceder al historial de estados
- Aislamiento entre threads

**Dominio**: Universo de Stephen King  
**Flujo**: Mismo grafo del Cap 02 con `checkpointer=MemorySaver()`

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from rich import print as rprint

DATA_PATH = Path("../00_datos/king_books.json")
king_books = json.loads(DATA_PATH.read_text(encoding="utf-8"))

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-5-mini"), temperature=0)
print("Dependencias cargadas ✓")

## 1. Sin Checkpointer: el grafo no recuerda

Sin checkpointer, cada invocación es independiente — el grafo no tiene memoria entre llamadas.

In [ ]:
KING_SYSTEM = """Eres un experto en Stephen King. Recuerdas lo que el usuario te ha contado anteriormente.
Si el usuario menciona algo que no sé, lo reconozco honestamente."""

def node_king_expert(state: MessagesState) -> dict:
    response = llm.invoke([SystemMessage(content=KING_SYSTEM)] + state["messages"])
    return {"messages": [response]}

# Grafo sin checkpointer
builder = StateGraph(MessagesState)
builder.add_node("experto", node_king_expert)
builder.add_edge(START, "experto")
builder.add_edge("experto", END)
graph_sin_memoria = builder.compile()

# Primera invocación
r1 = graph_sin_memoria.invoke({"messages": [HumanMessage(content="Me llamo Carlos y me interesa The Shining")]})
print("Turno 1:", r1["messages"][-1].content[:200])

# Segunda invocación — sin memoria
r2 = graph_sin_memoria.invoke({"messages": [HumanMessage(content="¿Cómo me llamo y qué libro mencioné?")]})
print("\nTurno 2 (sin memoria):", r2["messages"][-1].content[:200])
print("\n⚠️ El grafo no recuerda la primera conversación")

## 2. Con MemorySaver: el grafo recuerda por thread

Ahora añadimos `checkpointer=MemorySaver()` al compilar.  
Cada invocación necesita un `config={"configurable": {"thread_id": "..."}}`.  

In [ ]:
checkpointer = MemorySaver()

graph_con_memoria = builder.compile(checkpointer=checkpointer)

# Config para identificar la sesión
config = {"configurable": {"thread_id": "sesion_carlos"}}

# Primera vuelta
r1 = graph_con_memoria.invoke(
    {"messages": [HumanMessage(content="Me llamo Carlos y me interesa The Shining")]},
    config=config
)
print("Turno 1:", r1["messages"][-1].content[:200])

# Segunda vuelta — el grafo recuerda
r2 = graph_con_memoria.invoke(
    {"messages": [HumanMessage(content="¿Cómo me llamo y qué libro mencioné?")]},
    config=config
)
print("\nTurno 2 (con memoria):", r2["messages"][-1].content[:200])

## 3. Aislamiento entre Threads

Diferentes `thread_id` tienen conversaciones completamente independientes.

In [ ]:
config_alice = {"configurable": {"thread_id": "sesion_alice"}}
config_bob = {"configurable": {"thread_id": "sesion_bob"}}

# Alice habla de It
graph_con_memoria.invoke(
    {"messages": [HumanMessage(content="Estoy estudiando la novela It de King")]},
    config=config_alice
)

# Bob habla de Misery
graph_con_memoria.invoke(
    {"messages": [HumanMessage(content="Me interesa la psicología de Annie Wilkes en Misery")]},
    config=config_bob
)

# Alice pregunta sobre su tema
r_alice = graph_con_memoria.invoke(
    {"messages": [HumanMessage(content="¿Qué libro estoy estudiando?")]},
    config=config_alice
)
r_bob = graph_con_memoria.invoke(
    {"messages": [HumanMessage(content="¿De qué personaje hablé?")]},
    config=config_bob
)

print("Alice:", r_alice["messages"][-1].content[:200])
print("\nBob:", r_bob["messages"][-1].content[:200])
print("\n✅ Cada thread mantiene su propio contexto independiente")

## 4. `get_state_history`: Acceder al Historial de Checkpoints

In [ ]:
# Ver el historial de estados de la sesión de Alice
print("=== Historial de checkpoints para sesion_alice ===\n")
for i, checkpoint in enumerate(graph_con_memoria.get_state_history(config_alice)):
    n_messages = len(checkpoint.values.get("messages", []))
    print(f"Checkpoint {i}: {n_messages} mensajes | next: {checkpoint.next}")
    if i >= 4:
        print("  ...")
        break

## 5. Modo Book Club: Thread Compartido vs Privado

En un book club, varios lectores pueden compartir un thread (discusión grupal) o tener threads privados.

In [ ]:
# Thread compartido del club
config_club = {"configurable": {"thread_id": "book_club_it_2024"}}

# Miembro 1 añade su perspectiva
graph_con_memoria.invoke(
    {"messages": [HumanMessage(content="El miedo en It representa la pérdida de la inocencia infantil")]},
    config=config_club
)

# Miembro 2 añade su perspectiva al mismo thread
graph_con_memoria.invoke(
    {"messages": [HumanMessage(content="Yo creo que también representa el miedo colectivo de una comunidad")]},
    config=config_club
)

# Resumir la discusión grupal
r_resumen = graph_con_memoria.invoke(
    {"messages": [HumanMessage(content="Resume los temas que hemos discutido sobre It")]},
    config=config_club
)

print("Resumen del book club:")
print(r_resumen["messages"][-1].content[:400])

## Resumen del Capítulo

| Concepto | Descripción |
|---------|-------------|
| `MemorySaver` | Checkpointer in-memory (solo para desarrollo) |
| `thread_id` | Identifica sesiones de conversación |
| `config = {"configurable": {"thread_id": "..."}}` | Necesario en toda invocación con checkpointer |
| `get_state_history(config)` | Accede al historial de checkpoints de un thread |
| Aislamiento | Threads diferentes tienen estados completamente independientes |

⚠️ **Producción**: MemorySaver no persiste entre reinicios.  
Usar `langgraph-checkpoint-postgres` o similar en producción.

**Próximo capítulo**: Paralelización con `Send()` y fan-out